In [1]:
from google.colab import files

uploaded = files.upload()

Saving TSL_RSH_CURVATURE_GPU_CAMPAIGN.zip to TSL_RSH_CURVATURE_GPU_CAMPAIGN.zip


In [2]:
from pathlib import Path
import hashlib

zip_path = Path("/content/TSL_RSH_CURVATURE_GPU_CAMPAIGN.zip")
expected = "241ea1d30406f2874d9b52f2a5b4b202673551ec1b112bb5be6d63eae15290e3"

actual = hashlib.sha256(zip_path.read_bytes()).hexdigest()

print("Expected:", expected)
print("Actual:  ", actual)
assert actual == expected, "STOP: ZIP checksum mismatch"
print("ZIP CHECKSUM PASS")

Expected: 241ea1d30406f2874d9b52f2a5b4b202673551ec1b112bb5be6d63eae15290e3
Actual:   241ea1d30406f2874d9b52f2a5b4b202673551ec1b112bb5be6d63eae15290e3
ZIP CHECKSUM PASS


In [3]:
import zipfile
from pathlib import Path

dest = Path("/content/tsl_rsh_curvature")
dest.mkdir(exist_ok=True)

with zipfile.ZipFile("/content/TSL_RSH_CURVATURE_GPU_CAMPAIGN.zip") as z:
    z.extractall(dest)

print("Extracted to:", dest)

Extracted to: /content/tsl_rsh_curvature


In [4]:
from pathlib import Path

root = Path("/content/tsl_rsh_curvature")
runners = list(root.rglob("run_curvature_gpu_campaign.py"))

assert len(runners) == 1, runners

campaign_dir = runners[0].parent
print("Campaign directory:", campaign_dir)

for p in sorted(campaign_dir.iterdir()):
    print(p.name)

Campaign directory: /content/tsl_rsh_curvature
CURVATURE_ANALYSIS_PROTOCOL.json
CURVATURE_COORDINATE_DEFINITIONS.json
CURVATURE_GEOMETRY_MANIFEST.csv
CURVATURE_PAIR_PANEL.json
DISPLACEMENT_PROTOCOL.json
FROZEN_GPU_QM_PROTOCOL.json
GEOMETRY_CONSTRUCTION_AUDIT.json
PACKAGE_SHA256SUMS
finalize_curvature_package.py
geometries
prepare_curvature_campaign.py
run_curvature_gpu_campaign.py


In [5]:
!nvidia-smi

Sun Aug 23 15:01:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [6]:
%%bash
python -m pip install -q \
  pyscf==2.14.0 \
  gpu4pyscf-cuda12x==1.8.0 \
  dftd3==1.5.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.2/392.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.7/151.7 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 123.2 MB/s eta 0:00:00


In [7]:
import pyscf
import cupy as cp
from importlib.metadata import version

print("PySCF:", pyscf.__version__)
print("GPU4PySCF:", version("gpu4pyscf-cuda12x"))
print("dftd3:", version("dftd3"))
print("CuPy:", cp.__version__)
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])

assert pyscf.__version__ == "2.14.0"
assert version("gpu4pyscf-cuda12x") == "1.8.0"
assert version("dftd3") == "1.5.0"

PySCF: 2.14.0
GPU4PySCF: 1.8.0
dftd3: 1.5.0
CuPy: 14.0.1
GPU: b'NVIDIA A100-SXM4-40GB'


In [8]:
import csv, os
from pathlib import Path

os.chdir(campaign_dir)

rows = list(csv.DictReader(open("CURVATURE_GEOMETRY_MANIFEST.csv")))
print("Manifest points:", len(rows))
print("Unique IDs:", len({r["campaign_id"] for r in rows}))
print("Unique geometry hashes:", len({r["geometry_sha256"] for r in rows}))

assert len(rows) == 76
assert len({r["campaign_id"] for r in rows}) == 76
assert len({r["geometry_sha256"] for r in rows}) == 76

print("MANIFEST PREFLIGHT PASS")

Manifest points: 76
Unique IDs: 76
Unique geometry hashes: 76
MANIFEST PREFLIGHT PASS


In [9]:
import os
os.environ["CURVATURE_CAMPAIGN_EXECUTION_AUTHORIZED"] = "YES_AFTER_INDEPENDENT_REVIEW"
print("Execution authorized after independent review.")

Execution authorized after independent review.


In [10]:
!python run_curvature_gpu_campaign.py

Streaming output truncated to the last 5000 lines.
pruning grids: <function nwchem_prune at 0x788fa7f1b4c0>
grids dens level: 2
symmetrized grids: False
atomic radii adjust function: <function treutler_atomic_radii_adjust at 0x788fa7f1b600>
small_rho_cutoff = 0
Set gradient conv threshold to 0.0001
tot grids = 453120
init E= -1480.22954257579
  HOMO = -0.279707078567923  LUMO = -0.167611316794094  gap/eV = 3.05028
cycle= 1 E= -1477.32861726819  delta_E=  2.9  |g|= 1.51  |ddm|= 8.67
  HOMO = -0.0178677897202781  LUMO = -0.00446596487552089  gap/eV = 0.36468
cycle= 2 E= -1472.98836172793  delta_E= 4.34  |g|= 3.88  |ddm|= 9.38
  HOMO = -0.124444360574375  LUMO = -0.110848379022203  gap/eV = 0.36997
cycle= 3 E= -1476.17036665677  delta_E= -3.18  |g|= 1.97  |ddm|= 8.88
  HOMO = -0.195431741306056  LUMO = -0.0618624060880296  gap/eV = 3.63461
cycle= 4 E= -1477.71826102557  delta_E= -1.55  |g|= 0.684  |ddm|= 2.97
  HOMO = -0.167198417147121  LUMO = -0.124960356568114  gap/eV = 1.14936
cycle= 

In [11]:
from pathlib import Path
import json, hashlib, csv
import numpy as np

campaign_dir = Path("/content/tsl_rsh_curvature")
results_dir = campaign_dir / "curvature_gpu_results"
manifest = list(csv.DictReader((campaign_dir / "CURVATURE_GEOMETRY_MANIFEST.csv").open()))

assert len(manifest) == 76

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

completed = []
failed = []
problems = []

for row in manifest:
    cid = row["campaign_id"]
    d = results_dir / cid

    if not d.is_dir():
        failed.append(cid)
        continue

    try:
        r = json.loads((d / "result.json").read_text())

        assert r["status"] == "CONVERGED"
        assert r["campaign_id"] == cid
        assert r["geometry_sha256"] == row["geometry_sha256"]
        assert r["atom_count"] == 56
        assert r["charge"] == 0
        assert r["multiplicity"] == 1
        assert r["electron_count"] == 202
        assert r["scf_converged"] is True

        eg = np.loadtxt(d / "electronic_gradient_hartree_per_bohr.txt")
        dg = np.loadtxt(d / "d3_gradient_hartree_per_bohr.txt")
        tg = np.loadtxt(d / "total_gradient_hartree_per_bohr.txt")
        f  = np.loadtxt(d / "force_hartree_per_bohr.txt")

        assert eg.shape == dg.shape == tg.shape == f.shape == (56,3)
        assert np.isfinite(eg).all()
        assert np.isfinite(dg).all()
        assert np.isfinite(tg).all()
        assert np.isfinite(f).all()

        assert np.allclose(tg, eg + dg, rtol=0, atol=1e-12)
        assert np.allclose(f, -tg, rtol=0, atol=1e-12)

        # verify SHA256SUMS
        sums = (d / "SHA256SUMS").read_text().splitlines()
        for line in sums:
            expected, name = line.split("  ", 1)
            actual = sha256(d / name)
            assert actual == expected, f"{cid}: checksum mismatch {name}"

        completed.append(cid)

    except Exception as e:
        problems.append((cid, repr(e)))

failed_dirs = list(results_dir.glob("*.FAILED.*"))
partial_dirs = list(results_dir.glob(".*.partial.*"))

print("EXPECTED =", 76)
print("COMPLETED =", len(completed))
print("MISSING =", len(failed))
print("PROBLEMS =", len(problems))
print("FAILED_DIRS =", len(failed_dirs))
print("PARTIAL_DIRS =", len(partial_dirs))

if failed:
    print("Missing:", failed)
if problems:
    print("Problems:")
    for x in problems:
        print(x)

assert len(completed) == 76
assert not failed
assert not problems
assert not failed_dirs
assert not partial_dirs

print("FULL CURVATURE DATASET VERIFICATION PASS")

EXPECTED = 76
COMPLETED = 76
MISSING = 0
PROBLEMS = 0
FAILED_DIRS = 0
PARTIAL_DIRS = 0
FULL CURVATURE DATASET VERIFICATION PASS


In [12]:
import shutil, hashlib
from pathlib import Path

archive = shutil.make_archive(
    "/content/TSL_RSH_CURVATURE_GPU_RESULTS",
    "zip",
    root_dir="/content/tsl_rsh_curvature",
    base_dir="curvature_gpu_results"
)

sha = hashlib.sha256(Path(archive).read_bytes()).hexdigest()

print("ARCHIVE =", archive)
print("SHA256 =", sha)

ARCHIVE = /content/TSL_RSH_CURVATURE_GPU_RESULTS.zip
SHA256 = 732963b8682b966539cb2eadbe55f4fca9f181611364dc788698da220ac09cbf


In [13]:
from google.colab import files
files.download("/content/TSL_RSH_CURVATURE_GPU_RESULTS.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>